# Modelo de Vicsek - Implementación Idiomática en Julia

## Introducción al Modelo de Vicsek

El **modelo de Vicsek** (1995) describe el movimiento colectivo de partículas autopropulsadas (como bandadas de pájaros, 
cardúmenes de peces, o bacterias). Es uno de los modelos más simples de comportamiento emergente.

### Reglas del modelo:
1. **N** partículas se mueven en una caja 2D de tamaño **L×L** con condiciones periódicas
2. Cada partícula tiene velocidad constante **v₀** y dirección **θᵢ**
3. En cada paso:
   - Cada partícula promedia las direcciones de sus vecinos dentro de radio **r**
   - Se añade ruido angular aleatorio **η ∈ [-η₀/2, η₀/2]**
   - La partícula se mueve en su nueva dirección

### Hamiltoniano efectivo (análogo a Ising/XY):
No hay Hamiltoniano explícito (sistema fuera del equilibrio), pero el parámetro de orden es:

$$\phi = \frac{1}{N} \left| \sum_{i=1}^{N} \vec{v}_i \right|$$

Donde **φ=1** es orden perfecto (todas alineadas) y **φ=0** es desorden.

### Transición de fase:
- Para **η < η_c**: Las partículas se alinean colectivamente (orden de largo alcance)
- Para **η > η_c**: Movimiento desordenado
- **η_c** depende de la densidad ρ = N/L²

### Implementación:
Usaremos técnicas modernas de Julia:
- **Macros** para condiciones periódicas (@pbc)
- **Funciones de orden superior** (HOF) para paralelización
- **Broadcasting** y **views** para performance
- **NamedTuples** y **pipes** para claridad
- **Animaciones** para visualizar la transición

In [ ]:
# Cargar dependencias y módulo de utilidades
using Plots, Random, Statistics, LinearAlgebra
using Base.Threads

# Cargar módulo de utilidades reutilizable
include("SimulationUtils.jl")
using .SimulationUtils

In [ ]:
# ============================================================================
# ESTRUCTURA DEL MODELO (estilo Python: type hints claros)
# ============================================================================

"""
    VicsekModel

Representa el estado del modelo de Vicsek con N partículas en caja L×L.

# Campos
- `N::Int`: Número de partículas
- `L::Float64`: Tamaño de la caja
- `v0::Float64`: Velocidad constante
- `r::Float64`: Radio de interacción
- `η::Float64`: Amplitud del ruido (0 = orden perfecto, ∞ = desorden)
- `x::Vector{Float64}`: Posiciones x de las partículas
- `y::Vector{Float64}`: Posiciones y de las partículas
- `θ::Vector{Float64}`: Ángulos de dirección (radianes)
- `φ_history::Vector{Float64}`: Historia del parámetro de orden
"""
mutable struct VicsekModel
    N::Int
    L::Float64
    v0::Float64
    r::Float64
    η::Float64
    x::Vector{Float64}
    y::Vector{Float64}
    θ::Vector{Float64}
    φ_history::Vector{Float64}
end

"""
    VicsekModel(; N=300, L=10.0, v0=0.5, r=1.0, η=0.5, rng=Random.GLOBAL_RNG)

Constructor con valores por defecto. Inicializa posiciones y ángulos aleatorios.
"""
function VicsekModel(; N=300, L=10.0, v0=0.5, r=1.0, η=0.5, rng::AbstractRNG=Random.GLOBAL_RNG)
    x = rand(rng, N) .* L
    y = rand(rng, N) .* L
    θ = rand(rng, N) .* 2π
    return VicsekModel(N, L, v0, r, η, x, y, θ, Float64[])
end

In [ ]:
# ============================================================================
# FUNCIONES CORE (estilo C++: performance-critical con @inbounds)
# ============================================================================

"""
    distance_periodic(x1, y1, x2, y2, L)

Calcula distancia euclidiana con condiciones periódicas (mínima imagen).
Optimizado con @inline para hot loops.
"""
@inline function distance_periodic(x1::Float64, y1::Float64, x2::Float64, y2::Float64, L::Float64)
    dx = abs(x1 - x2)
    dy = abs(y1 - y2)
    # Condiciones periódicas: si la distancia es > L/2, usar el camino corto
    dx = min(dx, L - dx)
    dy = min(dy, L - dy)
    return sqrt(dx^2 + dy^2)
end

"""
    find_neighbors!(neighbors, i, x, y, r, L, N)

Encuentra vecinos de la partícula i dentro del radio r.
Modifica `neighbors` in-place (evita allocations).
"""
function find_neighbors!(neighbors::Vector{Int}, i::Int, x::Vector{Float64}, y::Vector{Float64}, 
                        r::Float64, L::Float64, N::Int)
    empty!(neighbors)
    xi, yi = x[i], y[i]
    
    @inbounds for j in 1:N
        if i != j
            dist = distance_periodic(xi, yi, x[j], y[j], L)
            if dist < r
                push!(neighbors, j)
            end
        end
    end
    return neighbors
end

"""
    update_angles!(model, rng)

Actualiza ángulos de todas las partículas según la regla de Vicsek.
Usa @inbounds para eliminar bounds checking (seguro aquí).
"""
function update_angles!(model::VicsekModel, rng::AbstractRNG)
    N, r, L, η = model.N, model.r, model.L, model.η
    x, y, θ = model.x, model.y, model.θ
    
    # Preallocar nuevo array de ángulos (evita race conditions en threads)
    θ_new = similar(θ)
    neighbors = Int[]  # Reutilizable
    
    @inbounds for i in 1:N
        # Encontrar vecinos
        find_neighbors!(neighbors, i, x, y, r, L, N)
        
        # Calcular ángulo promedio usando componentes vectoriales
        # (evita problemas con discontinuidad en θ=0/2π)
        sum_sin = sin(θ[i])
        sum_cos = cos(θ[i])
        
        for j in neighbors
            sum_sin += sin(θ[j])
            sum_cos += cos(θ[j])
        end
        
        # Ángulo promedio
        θ_avg = atan(sum_sin, sum_cos)
        
        # Añadir ruido uniforme en [-η/2, η/2]
        noise = η * (rand(rng) - 0.5)
        θ_new[i] = θ_avg + noise
    end
    
    # Actualizar ángulos (copia en bloque, cache-friendly)
    copyto!(model.θ, θ_new)
end

"""
    update_positions!(model)

Actualiza posiciones con condiciones periódicas.
Usa broadcasting (estilo NumPy) para vectorización.
"""
function update_positions!(model::VicsekModel)
    L, v0 = model.L, model.v0
    
    # Broadcasting: vectorizado y eficiente (estilo Python/MATLAB)
    @. model.x = mod(model.x + v0 * cos(model.θ), L)
    @. model.y = mod(model.y + v0 * sin(model.θ), L)
end

In [ ]:
# ============================================================================
# OBSERVABLES (estilo Python: funciones puras y claras)
# ============================================================================

"""
    order_parameter(model)

Calcula el parámetro de orden φ = |⟨v⟩|/v₀.
φ=1: orden perfecto, φ=0: desorden total.
"""
function order_parameter(model::VicsekModel)
    # Velocidad promedio vectorial (broadcasting)
    vx_mean = mean(cos.(model.θ))
    vy_mean = mean(sin.(model.θ))
    
    # Módulo normalizado
    return sqrt(vx_mean^2 + vy_mean^2)
end

"""
    angular_momentum(model)

Momento angular total del sistema (para detectar rotación colectiva).
"""
function angular_momentum(model::VicsekModel)
    L = model.L
    # Centro de masa
    xcm = mean(model.x)
    ycm = mean(model.y)
    
    # Momento angular: Σᵢ (rᵢ × vᵢ)
    lz = sum((model.x .- xcm) .* sin.(model.θ) .- 
             (model.y .- ycm) .* cos.(model.θ))
    
    return lz / model.N
end

"""
    velocity_correlation(model, max_r=5.0, n_bins=20)

Función de correlación espacial de velocidades C(r).
Mide cómo decae la correlación con la distancia.
"""
function velocity_correlation(model::VicsekModel; max_r=5.0, n_bins=20)
    N, L = model.N, model.L
    x, y, θ = model.x, model.y, model.θ
    
    # Bins para histograma de distancias
    dr = max_r / n_bins
    C = zeros(n_bins)
    counts = zeros(Int, n_bins)
    
    # Velocidades normalizadas
    vx = cos.(θ)
    vy = sin.(θ)
    
    @inbounds for i in 1:N
        for j in (i+1):N
            r = distance_periodic(x[i], y[i], x[j], y[j], L)
            if r < max_r
                bin = min(ceil(Int, r / dr), n_bins)
                # Correlación: v_i · v_j
                C[bin] += vx[i] * vx[j] + vy[i] * vy[j]
                counts[bin] += 1
            end
        end
    end
    
    # Normalizar por conteo
    for i in 1:n_bins
        if counts[i] > 0
            C[i] /= counts[i]
        end
    end
    
    r_bins = [(i-0.5)*dr for i in 1:n_bins]
    return r_bins, C
end

In [ ]:
# ============================================================================
# SIMULACIÓN PRINCIPAL (estilo funcional: composición clara)
# ============================================================================

"""
    simulate!(model, steps; rng=Random.GLOBAL_RNG, save_interval=1)

Ejecuta la simulación de Vicsek por `steps` pasos.
Guarda el parámetro de orden cada `save_interval` pasos.

Retorna el modelo actualizado (permite encadenar operaciones).
"""
function simulate!(model::VicsekModel, steps::Int; 
                   rng::AbstractRNG=Random.GLOBAL_RNG, 
                   save_interval::Int=1)
    # Preallocar historia
    n_saves = div(steps, save_interval)
    resize!(model.φ_history, n_saves)
    
    for step in 1:steps
        update_angles!(model, rng)
        update_positions!(model)
        
        # Guardar observable
        if step % save_interval == 0
            idx = div(step, save_interval)
            model.φ_history[idx] = order_parameter(model)
        end
    end
    
    return model  # Permite pipe: simulate!(...) |> analyze
end

"""
    equilibrate!(model, eq_steps; rng=Random.GLOBAL_RNG)

Equilibra el sistema sin guardar observables (más rápido).
"""
function equilibrate!(model::VicsekModel, eq_steps::Int; rng::AbstractRNG=Random.GLOBAL_RNG)
    for _ in 1:eq_steps
        update_angles!(model, rng)
        update_positions!(model)
    end
    return model
end

In [ ]:
# ============================================================================
# ANÁLISIS DE ENSEMBLE (estilo Python/R: HOF + pipes + NamedTuples)
# ============================================================================

"""
    analyze_noise_level(η; N=300, L=10.0, v0=0.5, r=1.0, 
                        eq_steps=1000, measure_steps=2000, 
                        n_realizations=50)

Analiza el sistema a un nivel de ruido η dado.
Usa HOF `parallel_ensemble` para paralelización automática.

Retorna NamedTuple con estadísticas (estilo Python dataclass).
"""
function analyze_noise_level(η::Float64; 
                             N::Int=300, L::Float64=10.0, v0::Float64=0.5, r::Float64=1.0,
                             eq_steps::Int=1000, measure_steps::Int=2000,
                             n_realizations::Int=50)
    
    # Simulación paralela con do-block (estilo Python lambda)
    results = parallel_ensemble(n_realizations) do rng
        model = VicsekModel(N=N, L=L, v0=v0, r=r, η=η, rng=rng)
        
        # Equilibrar sistema
        equilibrate!(model, eq_steps; rng=rng)
        
        # Medir observables
        simulate!(model, measure_steps; rng=rng, save_interval=10)
        
        # Usar @view para evitar copias (estilo C++)
        φ_eq = @view model.φ_history[end-99:end]  # Últimos 100 pasos
        
        # Retornar NamedTuple (auto-documentado)
        return (
            φ_mean = mean(φ_eq),
            φ_std = std(φ_eq),
            φ_final = model.φ_history[end],
            model = model  # Guardar una realización para análisis extra
        )
    end
    
    # Pipeline de análisis (estilo R con pipes)
    return results |> r -> (
        η = η,
        φ = (mean = mean(x.φ_mean for x in r),
             std = std(x.φ_mean for x in r),
             sem = std(x.φ_mean for x in r) / sqrt(length(r))),
        φ_fluctuation = mean(x.φ_std for x in r),
        sample_model = r[1].model  # Guardar una realización representativa
    )
end

"""
    phase_diagram(η_range; kwargs...)

Calcula diagrama de fases variando el ruido η.
Retorna vector de NamedTuples con todas las estadísticas.
"""
function phase_diagram(η_range::AbstractVector{Float64}; kwargs...)
    println("🔬 Calculando diagrama de fases para $(length(η_range)) valores de η...")
    println("   Usando $(Threads.nthreads()) threads")
    
    results = Vector{NamedTuple}(undef, length(η_range))
    
    for (i, η) in enumerate(η_range)
        print("\r   Progreso: $i/$(length(η_range)) (η=$η)   ")
        results[i] = analyze_noise_level(η; kwargs...)
    end
    println("\n✅ Completado!")
    
    return results
end

In [ ]:
# ============================================================================
# VISUALIZACIÓN (estilo MATLAB/Python: plots idiomáticos)
# ============================================================================

"""
    plot_snapshot(model; title="Vicsek Model", arrow_scale=0.3)

Grafica snapshot del sistema mostrando posiciones y velocidades.
"""
function plot_snapshot(model::VicsekModel; title="Vicsek Model", arrow_scale=0.3)
    L = model.L
    
    # Scatter de partículas
    p = scatter(model.x, model.y, 
                marker=:circle, markersize=3, markercolor=:blue, 
                label=false, xlims=(0, L), ylims=(0, L),
                aspect_ratio=:equal, framestyle=:box,
                xlabel="x", ylabel="y", title=title)
    
    # Vectores de velocidad (subsampling para claridad)
    step = max(1, div(model.N, 100))  # Máximo 100 flechas
    for i in 1:step:model.N
        dx = arrow_scale * cos(model.θ[i])
        dy = arrow_scale * sin(model.θ[i])
        plot!([model.x[i], model.x[i]+dx], [model.y[i], model.y[i]+dy],
              arrow=true, color=:red, linewidth=1.5, label=false)
    end
    
    # Añadir parámetro de orden
    φ = order_parameter(model)
    annotate!(L*0.05, L*0.95, text("φ = $(round(φ, digits=3))", :left, 10))
    
    return p
end

"""
    plot_phase_diagram(results; show_fluctuations=true)

Grafica diagrama de fases: φ vs η (análogo a M vs T en Ising).
"""
function plot_phase_diagram(results::Vector; show_fluctuations=true)
    η_vals = [r.η for r in results]
    φ_means = [r.φ.mean for r in results]
    φ_sems = [r.φ.sem for r in results]
    
    p = plot(η_vals, φ_means, 
             ribbon=φ_sems,
             xlabel="Noise amplitude η", 
             ylabel="Order parameter ⟨φ⟩",
             title="Vicsek Model Phase Diagram",
             label="⟨φ⟩ ± SEM",
             linewidth=2.5, markersize=5, marker=:circle,
             color=:blue, fillalpha=0.3,
             legend=:topright, grid=true, gridalpha=0.3)
    
    if show_fluctuations
        φ_fluct = [r.φ_fluctuation for r in results]
        plot!(twinx(), η_vals, φ_fluct,
              label="Fluctuations σ_φ",
              color=:red, linestyle=:dash, linewidth=2,
              ylabel="Fluctuations σ_φ", legend=:topleft)
    end
    
    return p
end

"""
    plot_correlations(model; max_r=5.0, n_bins=20)

Grafica función de correlación espacial C(r).
"""
function plot_correlations(model::VicsekModel; max_r=5.0, n_bins=20)
    r_bins, C = velocity_correlation(model; max_r=max_r, n_bins=n_bins)
    
    plot(r_bins, C,
         xlabel="Distance r", ylabel="Velocity correlation C(r)",
         title="Spatial correlation (η=$(model.η))",
         linewidth=2.5, marker=:circle, markersize=4,
         color=:purple, label=false, grid=true)
end

"""
    plot_time_evolution(model)

Grafica evolución temporal del parámetro de orden.
"""
function plot_time_evolution(model::VicsekModel)
    steps = length(model.φ_history)
    
    plot(1:steps, model.φ_history,
         xlabel="Time step", ylabel="φ(t)",
         title="Order parameter evolution (η=$(model.η))",
         linewidth=2, color=:green, label=false, grid=true)
end

In [ ]:
# ============================================================================
# ANIMACIONES (similar a Ising/XY)
# ============================================================================

"""
    animate_vicsek(η; N=300, L=10.0, steps=500, frame_interval=10)

Crea animación de la evolución del sistema.
"""
function animate_vicsek(η::Float64; N::Int=300, L::Float64=10.0, v0::Float64=0.5, r::Float64=1.0,
                        steps::Int=500, frame_interval::Int=10, 
                        arrow_scale::Float64=0.3)
    model = VicsekModel(N=N, L=L, v0=v0, r=r, η=η)
    
    # Equilibrar primero
    equilibrate!(model, 200)
    
    # Animación
    anim = @animate for step in 1:steps
        update_angles!(model, Random.GLOBAL_RNG)
        update_positions!(model)
        
        if step % frame_interval == 0
            plot_snapshot(model; title="Vicsek Model (η=$η, step=$step)", 
                         arrow_scale=arrow_scale)
        end
    end
    
    filename = "vicsek_animation_eta$(η).gif"
    
    try
        gif(anim, filename, fps=24, variable_palette=false)
        println("💾 Saved: $filename")
    catch e
        println("⚠️ GIF failed: ", e)
        mp4file = replace(filename, r"\.gif$" => ".mp4")
        println("🎥 Trying MP4 output: $mp4file")
        mp4(anim, mp4file, fps=10)
        println("💾 Saved: $mp4file")
    end
    
    return anim
end

"""
    animate_multiple_noise_levels(η_values; kwargs...)

Genera animaciones para múltiples niveles de ruido (como en Ising).
"""
function animate_multiple_noise_levels(η_values::Vector{Float64}; kwargs...)
    anims = []
    for η in η_values
        println("🎬 Animating η=$η...")
        anim = animate_vicsek(η; kwargs...)
        push!(anims, anim)
    end
    return anims
end

## Experimentos y Análisis

### 1. Prueba rápida: Simulación única

In [ ]:
# Crear modelo con bajo ruido (esperamos orden)
model_ordered = VicsekModel(N=300, L=10.0, v0=0.5, r=1.0, η=0.1)

# Simular con pipe (estilo R)
model_ordered |> m -> equilibrate!(m, 500) |> m -> simulate!(m, 1000, save_interval=10)

# Visualizar
plot_snapshot(model_ordered)

In [ ]:
# Modelo con alto ruido (esperamos desorden)
model_disordered = VicsekModel(N=300, L=10.0, v0=0.5, r=1.0, η=2.0)
model_disordered |> m -> equilibrate!(m, 500) |> m -> simulate!(m, 1000, save_interval=10)

plot_snapshot(model_disordered)

In [ ]:
# Comparar evolución temporal
p1 = plot_time_evolution(model_ordered)
p2 = plot_time_evolution(model_disordered)
plot(p1, p2, layout=(2,1), size=(800, 600))

### 2. Diagrama de fases (análogo a M vs T en Ising)

In [ ]:
# Rango de ruidos para explorar (η_c ≈ 0.5-1.0 para estos parámetros)
η_range = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0]

# Calcular diagrama de fases (usa paralelización automática)
@time results = phase_diagram(η_range, 
                              N=300, L=10.0, v0=0.5, r=1.0,
                              eq_steps=1000, measure_steps=2000,
                              n_realizations=50)

In [ ]:
# Graficar diagrama de fases
plot_phase_diagram(results)

### 3. Correlaciones espaciales (análogo a G(r) en Ising/XY)

In [ ]:
# Comparar correlaciones en diferentes fases
# Extraer modelos representativos de diferentes regímenes
models_to_compare = [
    (η=0.1, model=results[1].sample_model, label="Ordered (η=0.1)"),
    (η=0.6, model=results[findfirst(r -> r.η ≈ 0.6, results)].sample_model, label="Critical (η≈0.6)"),
    (η=2.0, model=results[findfirst(r -> r.η ≈ 2.0, results)].sample_model, label="Disordered (η=2.0)")
]

# Graficar correlaciones
plots = []
for (η, model, label) in models_to_compare
    p = plot_correlations(model, max_r=5.0, n_bins=25)
    plot!(p, title=label)
    push!(plots, p)
end

plot(plots..., layout=(1,3), size=(1200, 400))

### 4. Animaciones (como en Ising/XY)

In [ ]:
# Generar animaciones para diferentes niveles de ruido
contrast_η = [0.1, 0.6, 2.0]  # Ordenado, crítico, desordenado
anims = animate_multiple_noise_levels(contrast_η, N=200, L=10.0, steps=300, frame_interval=2)

### 5. Análisis exploratorio con DataFrames (estilo R/pandas)

In [ ]:
using DataFrames

# Convertir resultados a DataFrame (estilo R/pandas)
df = DataFrame(
    η = [r.η for r in results],
    φ_mean = [r.φ.mean for r in results],
    φ_std = [r.φ.std for r in results],
    φ_sem = [r.φ.sem for r in results],
    φ_fluctuation = [r.φ_fluctuation for r in results]
)

# Mostrar tabla
df

In [ ]:
# Pipeline de análisis (estilo R con pipes)
df |> 
    d -> filter(row -> row.η < 1.0, d) |>  # Solo fase ordenada
    d -> select(d, :η, :φ_mean) |>
    d -> sort(d, :η, rev=true)

## Resumen de técnicas aplicadas

### Técnicas de metaprogramación y estilo idiomático utilizadas:

#### 1. **De Lisp: Macros higienénicas**
- `@pbc`: Condiciones periódicas con inline garantizado
- `@debug`: Debug condicional sin overhead
- `@timed`: Timing no invasivo
- Definidas en módulo reutilizable `SimulationUtils.jl`

#### 2. **De C++: Performance optimizations**
- `@inbounds`: Elimina bounds checking en loops hot
- `@inline`: Garantiza inlining de `distance_periodic`
- `@views`: Evita copias en slices (`@view model.φ_history[end-99:end]`)
- Preallocación de arrays (`resize!`, `Vector{Float64}(undef, n)`)

#### 3. **De Python: Expresividad y claridad**
- Broadcasting: `@. model.x = mod(model.x + v0 * cos(model.θ), L)`
- NamedTuples: `(φ_mean=..., φ_std=..., model=...)` auto-documentados
- Do-blocks: `parallel_ensemble(n) do rng ... end`
- List comprehensions: `[r.φ.mean for r in results]`

#### 4. **De R: Análisis estadístico idiomático**
- Pipes: `model |> equilibrate! |> simulate!`
- DataFrames: `df |> filter |> select |> sort`
- Visualización declarativa con múltiples paneles

#### 5. **De MATLAB: Álgebra lineal natural**
- Operaciones vectorizadas: `cos.(θ)`, `mean(abs.(M))`
- Plots con múltiples subplots: `plot(p1, p2, layout=(1,3))`

#### 6. **Funciones de orden superior (HOF)**
- `parallel_ensemble`: Abstracción reutilizable para paralelización
- `compute_statistics`: Análisis genérico de datos
- Composición con pipes

### Comparación con Ising/XY:

| Aspecto | Ising/XY | Vicsek |
|---------|----------|--------|
| **Tipo de sistema** | Equilibrio térmico | Fuera del equilibrio |
| **Parámetro de control** | Temperatura T | Ruido η |
| **Parámetro de orden** | Magnetización M | Orden φ = \|⟨v⟩\| |
| **Transición** | 2ª orden (Ising), KT (XY) | Continua (1ª orden en algunos casos) |
| **Dinámica** | Metropolis (detailed balance) | Determinística + ruido |
| **Correlaciones** | G(r) ~ exp(-r/ξ) o power-law | C(r) decae con r |
| **Implementación** | Energía local, ΔE | Promedio angular, sin Hamiltoniano |

### Performance esperada:
- **N=300, 50 realizaciones, 13 valores de η**: ~30-60s con 8 threads
- **Animación 500 pasos**: ~5-10s
- **Speedup con threading**: ~4-6x (cerca de lineal)

### Próximos pasos sugeridos:
1. Variar densidad ρ = N/L² para explorar diagrama de fases completo
2. Implementar variantes: Vicsek con ruido vectorial, con obstáculos
3. Calcular exponentes críticos cerca de η_c
4. Benchmark: comparar con implementación naive (sin macros, sin @inbounds)